# FarmSense Fine-tuning v2
## 133 exemples — corrections prix, photos, Wolof
## Executer dans l ordre : 1 → 2 → 3 → 4
### GPU T4 requis

In [ ]:
# CELLULE 1 — Installation et dataset
import os
os.system('pip install -q unsloth trl transformers datasets peft accelerate bitsandbytes')
print('OK packages')

GITHUB_URL = 'https://github.com/ndaosaer/farmsense'
if not os.path.exists('/kaggle/working/farmsense'):
    os.system(f'git clone {GITHUB_URL} /kaggle/working/farmsense')
else:
    os.system('git -C /kaggle/working/farmsense pull')
print('OK repo')

os.chdir('/kaggle/working/farmsense/training')
os.system('python3 generate_dataset.py')
os.system('python3 generate_dataset_v2.py')
os.system('python3 generate_dataset_v3.py')
os.system('cat farmsense_dataset.jsonl farmsense_dataset_v2.jsonl farmsense_dataset_v3.jsonl > farmsense_dataset_v4.jsonl')

import json
data = [json.loads(l) for l in open('farmsense_dataset_v4.jsonl')]
print(f'OK dataset v4 : {len(data)} exemples')

In [ ]:
# CELLULE 2 — Charger le modele et configurer LoRA
import os, torch, gc
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-4-E4B-it',
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
    device_map={'': 0},
)
print('OK modele de base charge')

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'OK LoRA — parametres entrainables : {model.num_parameters(only_trainable=True):,}')

In [ ]:
# CELLULE 3 — Fine-tuning
import json, torch, gc
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

gc.collect()
torch.cuda.empty_cache()

examples = [json.loads(l) for l in open('/kaggle/working/farmsense/training/farmsense_dataset_v4.jsonl')]
print(f'Dataset : {len(examples)} exemples')

def format_example(ex):
    return {'text': tokenizer.apply_chat_template(ex['conversations'], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(examples).map(format_example, remove_columns=['conversations'])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/kaggle/working/farmsense-gemma4-v2',
        report_to='none',
        dataloader_pin_memory=False,
        dataset_text_field='text',
        max_seq_length=1024,
    ),
)

print('Demarrage fine-tuning v2...')
stats = trainer.train()
print(f'Termine ! Duree : {stats.metrics["train_runtime"]:.0f}s')

In [ ]:
# CELLULE 4 — Tests et publication HuggingFace
from unsloth import FastLanguageModel
import torch

HF_TOKEN = 'hf_TON_TOKEN'  # <- remplacer
HF_REPO  = 'ndaosaer/farmsense-gemma4-v2'

FastLanguageModel.for_inference(model)

def test(prompt):
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': 'Tu es FarmSense. Sans markdown, max 8 lignes.'}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': prompt}]}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=200, temperature=0.2,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print('=== Test 1 : Maladie ===')
print(test('[Zone: Kaolack | Langue: Français]\nMes feuilles de mil jaunissent depuis le bas.'))
print()
print('=== Test 2 : Prix ===')
print(test('[Zone: Kaolack | Langue: Français]\nPrix de larachide cette saison ?'))
print()
print('=== Test 3 : Wolof ===')
print(test('[Zone: Kaolack | Langue: Wolof]\nFeuilles yu mil dafa set ci jaambur.'))
print()
print('=== Test 4 : Photo non-agricole ===')
print(test('[Zone: Dakar | Langue: Français]\nAnalyse cette photo.'))

print('\nPublication HuggingFace...')
model.push_to_hub(HF_REPO, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f'OK : https://huggingface.co/{HF_REPO}')